In [ ]:
# notebooks/exploration.ipynb
import yaml
import numpy as np
from nuscenes.nuscenes import NuScenes
import os
import sys

notebook_path = os.path.abspath(os.getcwd())
project_root = os.path.dirname(notebook_path)
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added project root to sys.path: {project_root}")

from src.core import DepthImage, DepthImageLibrary 
from src.data_utils.nuscenes_helper import get_lidar_sweep_data
from src.utils.visualization import plot_lidar_sweep_with_depth_image_pixels

# For development, you might want to enable autoreload
%load_ext autoreload
%autoreload 2


In [ ]:

def run_single_sweep_test(config_path: str):
    # Load configuration
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Initialize NuScenes
    nusc = NuScenes(version=config['nuscenes']['version'], dataroot=config['nuscenes']['dataroot'], verbose=True)

    # --- Select a sample LiDAR sweep ---
    # Option 1: Use a predefined token from config (if set)
    lidar_token = config['nuscenes'].get('lidar_token_to_load')
    
    # Option 2: Or get the first LiDAR sweep from a specific scene or the first scene
    if not lidar_token:
        scene_name_to_load = config['nuscenes'].get('scene_name_to_load')
        if scene_name_to_load:
            scenes_by_name = {s['name']: s for s in nusc.scene}
            if scene_name_to_load not in scenes_by_name:
                print(f"Error: Scene '{scene_name_to_load}' not found. Available scenes: {list(scenes_by_name.keys())[:5]}")
                return
            scene_token = scenes_by_name[scene_name_to_load]['token']
        else: # Default to the first scene in the dataset
            scene_token = nusc.scene[0]['token']
        
        scene_info = nusc.get('scene', scene_token)
        first_sample_token = scene_info['first_sample_token']
        sample = nusc.get('sample', first_sample_token)
        lidar_token = sample['data']['LIDAR_TOP']
        print(f"Using LiDAR token: {lidar_token} from scene: {nusc.get('scene', scene_token)['name']}")

    # Load LiDAR data and pose for the selected sweep
    points_lidar_frame, T_global_lidar, lidar_timestamp = get_lidar_sweep_data(nusc, lidar_token)
    print(f"Loaded {points_lidar_frame.shape[0]} points for sweep at {lidar_timestamp:.2f}s.")
    print(f"LiDAR global pose (T_global_lidar):\n{T_global_lidar}")

    # Create a DepthImage instance
    # The pose for the DepthImage is the LiDAR's global pose at the time of the sweep
    di = DepthImage(image_pose_global=T_global_lidar, config=config, timestamp=lidar_timestamp)
    print(str(di))

    # Add points to the DepthImage
    # Points are in LiDAR frame, but DepthImage.add_point expects global points.
    # So, transform points_lidar_frame to global frame first.
    points_lidar_frame_h = np.hstack((points_lidar_frame, np.ones((points_lidar_frame.shape[0], 1)))) # Nx4
    points_global_h = (T_global_lidar @ points_lidar_frame_h.T).T # Nx4
    points_global = points_global_h[:, :3] # Nx3

    # Filter points by range before adding to depth image
    max_range = config['filtering']['max_point_range_meters']
    min_range = config['filtering']['min_point_range_meters']
    
    for pt_idx, pt_g in enumerate(points_global):
        # Calculate range from LiDAR origin (which is T_global_lidar[:3,3])
        # Or, more simply, use range from original points_lidar_frame
        range_val = np.linalg.norm(points_lidar_frame[pt_idx])
        if min_range <= range_val <= max_range:
            di.add_point(pt_g, label="non_event") # Initially all points are non_event

    print(f"After adding points to DepthImage: {str(di)}")

    # Visualize (optional)
    # For visualization, we can pass points in the LiDAR frame as that's often
    # the natural frame for the k3d camera setup.
    plot_lidar_sweep_with_depth_image_pixels(
        points_lidar_frame, # Pass original points in LiDAR frame for plotting
        di,
        point_size=0.05,
        max_range=max_range
    )

config_file_path = '../config/m_detector_config.yaml' 
run_single_sweep_test(config_file_path)

In [ ]:
def run_m_detector_simulation(config_path: str):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    nusc = NuScenes(version=config['nuscenes']['version'],
                    dataroot=config['nuscenes']['dataroot'])

    # --- Initialize Depth Image Library ---
    library_max_size = config['depth_image']['library_size']
    num_initial_sweeps_for_map = config['initialization']['num_initial_sweeps_for_map'] # Renamed for clarity
    depth_image_library = DepthImageLibrary(max_size=library_max_size)

    # --- Get a sequence of LiDAR tokens to process ---
    # For this example, let's just process a few sweeps from the first scene
    scene_token = nusc.scene[0]['token']
    current_sample_token = nusc.get('scene', scene_token)['first_sample_token']
    lidar_sensor_name = "LIDAR_TOP" # e.g., 'LIDAR_TOP'

    sweeps_processed = 0
    max_sweeps_to_process = num_initial_sweeps_for_map + 5 # Process a few more after initialization

    while current_sample_token and sweeps_processed < max_sweeps_to_process:
        sample = nusc.get('sample', current_sample_token)
        lidar_token = sample['data'][lidar_sensor_name]

        print(f"\n--- Processing sweep {sweeps_processed + 1} (token: {lidar_token}) ---")

        points_lidar_frame, T_global_lidar, lidar_timestamp = get_lidar_sweep_data(nusc, lidar_token)

        # --- 1. Create and Populate Current Depth Image ---
        # Assuming T = sweep duration for now, so one DI per sweep
        current_di = DepthImage(
            image_pose_global=T_global_lidar,
            config=config,
            timestamp=lidar_timestamp
        )

        # Add points to the current depth image
        points_lidar_frame_h = np.hstack((points_lidar_frame, np.ones((points_lidar_frame.shape[0], 1))))
        # Transform points to global frame before adding to DI, as DI expects global points
        points_global_h = (T_global_lidar @ points_lidar_frame_h.T).T
        points_global = points_global_h[:, :3]

        max_range = config['filtering']['max_point_range_meters']
        min_range = config['filtering']['min_point_range_meters']

        for pt_idx, pt_g in enumerate(points_global):
            range_val = np.linalg.norm(points_lidar_frame[pt_idx]) # Range check in sensor frame
            if min_range <= range_val <= max_range:
                # During initialization, all points are non-event
                current_di.add_point(pt_g, label="non_event") # Label will come from Phase 4 later

        print(f"Created DepthImage: {str(current_di)}")

        # --- 2. Add to Depth Image Library ---
        depth_image_library.add_image(current_di)
        print(f"DepthImageLibrary status: {str(depth_image_library)}")

        # --- 3. M-Detector Logic (Phases 2, 3, 4 would go here) ---
        if depth_image_library.is_ready_for_processing(num_initial_sweeps_for_map):
            print("Library is ready. M-Detector processing can begin.")
            latest_di_from_library = depth_image_library.get_latest_image()
            all_dis_in_library = depth_image_library.get_all_images()

            # Example: Access the one before the latest (if it exists)
            # prev_di = depth_image_library.get_image_by_index(-2)
            # if prev_di:
            # print(f"Previous DI timestamp: {prev_di.timestamp}")

            # TODO: Implement Phase 2 (Occlusion Likelihood)
            # TODO: Implement Phase 3 (Map Consistency)
            # TODO: Implement Phase 4 (Event Detection)
            # TODO: Implement Phase 5 (Event Tracking - optional)
            pass
        else:
            print(f"Initialization phase: {len(depth_image_library)} / {num_initial_sweeps_for_map} images collected.")

        # Move to the next sample
        if sample['next']:
            current_sample_token = sample['next']
        else:
            break # End of scene
        sweeps_processed += 1

config_file = '../config/m_detector_config.yaml'
run_m_detector_simulation(config_file)